# NẠP DỮ LIỆU THÔ ĐÃ QUY ĐỔI NHÃN

In [1]:
import os
import re
import sys
import pandas as pd
import numpy as np

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
if not os.environ.get('TEMP'):
    os.environ['TEMP'] = os.path.join(ROOT, '.tmp_run')
os.makedirs(os.environ['TEMP'], exist_ok=True)
os.environ['TMP'] = os.environ['TEMP']

In [3]:
#Đọc 3 tập dữ liệu từ bước 00
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_raw.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_raw.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_raw.csv'))

In [4]:
df_train.shape, df_dev.shape, df_test.shape

((7786, 13), (1112, 13), (2224, 13))

In [5]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 7786 entries, 0 to 7785
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   index         7786 non-null   int64
 1   sentence      7786 non-null   str  
 2   n_star        7786 non-null   int64
 3   date_time     7786 non-null   str  
 4   label         7786 non-null   str  
 5   sentiment     7786 non-null   int64
 6   aspect_group  7786 non-null   int64
 7   asp_0         7786 non-null   int64
 8   asp_1         7786 non-null   int64
 9   asp_2         7786 non-null   int64
 10  asp_3         7786 non-null   int64
 11  asp_4         7786 non-null   int64
 12  all_aspects   7786 non-null   str  
dtypes: int64(9), str(4)
memory usage: 2.9 MB


In [6]:
df_train.head()

,index,sentence,n_star,date_time,label,sentiment,aspect_group,asp_0,asp_1,asp_2,asp_3,asp_4,all_aspects
0,0,Mới mua máy này Tại thegioididong thốt nốt cảm...,5,2 tuần trước,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...,2,2,1,1,1,1,1,"0,1,2,3,4"
1,1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,5,14/09/2019,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};,1,2,0,0,1,0,1,"2,4"
2,2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,3,17/08/2020,{FEATURES#Negative};,0,0,1,0,0,0,0,0
3,3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",3,29/02/2020,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...,1,0,1,0,1,0,1,"0,2,4"
4,4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",5,4/6/2020,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...,2,3,1,0,1,1,0,"0,2,3"


## LÀM SẠCH VĂN BẢN, ẨN DANH PII VÀ TÁCH TỪ PYVI

In [7]:
#Nạp hàm chuẩn hóa giữ hoa/thường, lọc câu rỗng và tách từ pyvi từ src
from src.text_cleaner import clean_text, check_no_content, segment_words

In [8]:
#Thử nghiệm trên 1 câu mẫu chứa PII, tiếng cười, từ tiếng Anh lặp chữ và từ có 'ưu'
tempt = 'TGDĐ đt đẹppp quáaaa, lưu ý freee full app call kkk màu cam affordable ORD-2024-001 sđt 090.123.4567 tại thegioididong.com ❤️'
tempt

'TGDĐ đt đẹppp quáaaa, lưu ý freee full app call kkk màu cam affordable ORD-2024-001 sđt 090.123.4567 tại thegioididong.com ❤️'

In [9]:
#Làm sạch văn bản giữ hoa/thường cho PhoBERT
tempt_clean = clean_text(tempt, lowercase=False)
tempt_clean

'Thế Giới Di Động điện thoại đẹp quá, lưu ý free full app call cười màu cam affordable [ORDER_ID] sđt [PHONE] tại [URL] yêu_thích'

In [10]:
#Kiểm tra tách từ pyvi có bảo toàn nguyên dấu ngoặc [PHONE], [ORDER_ID], [URL]
segment_words(tempt_clean)

'Thế_Giới_Di_Động điện_thoại đẹp quá , lưu_ý free full app call cười màu cam affordable [ORDER_ID] sđt [PHONE] tại [URL] yêu_thích'

In [11]:
#Kiểm tra hàm lọc câu rỗng hoặc săn xu trên mẫu thử
check_no_content('cho 5 sao nhận xu'), check_no_content('Máy dùng tốt'), check_no_content(tempt_clean)

(True, False, False)

## ÁP DỤNG TIỀN XỬ LÝ CHO 3 TẬP DỮ LIỆU

In [12]:
#Áp dụng làm sạch (cased), gắn cờ câu vô nghĩa và tách từ pyvi
for df in [df_train, df_dev, df_test]:
    df['sentence_clean'] = df['sentence'].apply(lambda x: clean_text(x, lowercase=False))
    df['is_no_content'] = df['sentence_clean'].apply(check_no_content)
    df['sentence_seg'] = df['sentence_clean'].apply(segment_words)

In [13]:
df_train['is_no_content'].value_counts()

is_no_content
False    7786
Name: count, dtype: int64

In [14]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 7786 entries, 0 to 7785
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   index           7786 non-null   int64
 1   sentence        7786 non-null   str  
 2   n_star          7786 non-null   int64
 3   date_time       7786 non-null   str  
 4   label           7786 non-null   str  
 5   sentiment       7786 non-null   int64
 6   aspect_group    7786 non-null   int64
 7   asp_0           7786 non-null   int64
 8   asp_1           7786 non-null   int64
 9   asp_2           7786 non-null   int64
 10  asp_3           7786 non-null   int64
 11  asp_4           7786 non-null   int64
 12  all_aspects     7786 non-null   str  
 13  sentence_clean  7786 non-null   str  
 14  is_no_content   7786 non-null   bool 
 15  sentence_seg    7786 non-null   str  
dtypes: bool(1), int64(9), str(6)
memory usage: 6.1 MB


In [15]:
df_train[['sentence', 'sentence_clean', 'sentence_seg', 'is_no_content']].head()

,sentence,sentence_clean,sentence_seg,is_no_content
0,Mới mua máy này Tại thegioididong thốt nốt cảm...,Mới mua máy này Tại thegioididong thốt nốt cảm...,Mới mua máy này Tại thegioididong thốt_nốt cảm...,False
1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,Pin kém còn lại miễn chê mua 8 / 3 / 2019 tình...,False
2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,Sao lúc gọi điện_thoại màn_hình bị chấm nhỏ nh...,False
3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...","Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...","Mọi người cập_nhật phần_mềm lại , nó sẽ bớt tố...",False
4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...","Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...","Mới mua Sài được 1 tháng thấy pin rất trâu , S...",False


## CHỐNG TRÙNG LẶP NỘI BỘ VÀ CHỐNG RÒ RỈ GIỮA CÁC TẬP

In [16]:
#Loại bỏ các dòng trùng sentence_clean nội bộ trong tập train
n_tr_raw = len(df_train)
df_train = df_train.drop_duplicates(subset=['sentence_clean'], keep='first').reset_index(drop=True)
print('Train internal dupes dropped:', n_tr_raw - len(df_train))

Train internal dupes dropped: 2


In [17]:
#Xử lý trùng lặp nội bộ tập dev: xóa cả cặp nếu mâu thuẫn nhãn, giữ đầu nếu trùng nhãn
n_de_raw = len(df_dev)
lbl_cols = ['sentiment', 'aspect_group', 'all_aspects']
grp_n = df_dev.groupby('sentence_clean')[lbl_cols].transform('nunique').max(axis=1)
df_dev = df_dev[grp_n == 1].drop_duplicates(subset=['sentence_clean'], keep='first').reset_index(drop=True)
print('Dev internal conflicting/dup dropped:', n_de_raw - len(df_dev))

Dev internal conflicting/dup dropped: 2


In [18]:
#Xóa mọi dòng trong train có sentence_clean trùng với dev hoặc test để chống leakage 0%
s_dev = set(df_dev['sentence_clean'].fillna(''))
s_test = set(df_test['sentence_clean'].fillna(''))
drop_dev = int(df_train['sentence_clean'].isin(s_dev).sum())
drop_test = int(df_train['sentence_clean'].isin(s_test).sum())
df_train = df_train[~df_train['sentence_clean'].isin(s_dev | s_test)].reset_index(drop=True)
print('Train overlap dropped -> dev:', drop_dev, '| test:', drop_test, '| train remaining:', len(df_train))

Train overlap dropped -> dev: 3 | test: 5 | train remaining: 7776


In [19]:
#Xác minh 0 trùng lặp chéo và 0 trùng lặp nội bộ trên train/dev
len(set(df_train['sentence_clean']) & s_dev), len(set(df_train['sentence_clean']) & s_test), df_train['sentence_clean'].duplicated().sum(), df_dev['sentence_clean'].duplicated().sum()

(0, 0, np.int64(0), np.int64(0))

## KIỂM ĐỊNH TRƯỚC/SAU LÀM SẠCH (TOKEN DIFF AUDIT)

In [20]:
#So sánh số lỗi regex phổ biến trên toàn bộ 3 tập trước và sau khi qua src.text_cleaner
all_raw = pd.concat([df_train['sentence'], df_dev['sentence'], df_test['sentence']], ignore_index=True).fillna('')
all_clean = pd.concat([df_train['sentence_clean'], df_dev['sentence_clean'], df_test['sentence_clean']], ignore_index=True).fillna('')
all_seg = pd.concat([df_train['sentence_seg'], df_dev['sentence_seg'], df_test['sentence_seg']], ignore_index=True).fillna('')
print('luu/lu broken (lư ý|hữ) -> raw:', int(all_raw.str.contains(r'(?<!\w)(?:lư ý|hữ)(?!\w)', case=False).sum()), '| clean:', int(all_clean.str.contains(r'(?<!\w)(?:lư ý|hữ)(?!\w)', case=False).sum()))
print('full/app/call broken (ful|ap|cal) -> raw:', int(all_raw.str.contains(r'(?<!\w)(?:ful|ap|cal)(?!\w)', case=False).sum()), '| clean:', int(all_clean.str.contains(r'(?<!\w)(?:ful|ap|cal)(?!\w)', case=False).sum()))
print('tgdd/tgdđ unnormalized in clean:', int(all_clean.str.contains(r'(?<!\w)tgdđ?(?!\w)', case=False).sum()))

luu/lu broken (lư ý|hữ) -> raw: 0 | clean: 0


full/app/call broken (ful|ap|cal) -> raw: 5 | clean: 5


tgdd/tgdđ unnormalized in clean: 0


In [21]:
#Kiểm tra tiếp kkk, affordable, ngoặc [PHONE] sau pyvi và bộ đếm is_no_content
print('kkk remaining in clean:', int(all_clean.str.contains(r'(?<!\w)k{3,}(?!\w)', case=False).sum()))
print('affordable broken (af [ORDER_ID]):', int(all_clean.str.contains(r'af\s*\[ORDER_ID\]', case=False).sum()))
print('broken PII brackets in seg ([ PHONE ]):', int(all_seg.str.contains(r'\[\s+(?:URL|EMAIL|ORDER_ID|PHONE|PERSON)|(?:URL|EMAIL|ORDER_ID|PHONE|PERSON)\s+\]').sum()))
print('is_no_content counts (train/dev/test):', int(df_train['is_no_content'].sum()), int(df_dev['is_no_content'].sum()), int(df_test['is_no_content'].sum()))

kkk remaining in clean: 0
affordable broken (af [ORDER_ID]): 0
broken PII brackets in seg ([ PHONE ]): 0
is_no_content counts (train/dev/test): 0 0 0


## LƯU CÁC TẬP DỮ LIỆU ĐÃ LÀM SẠCH

In [22]:
#Xuất 3 tập dữ liệu sạch ra thư mục processed
df_train.to_csv(os.path.join(PROC_DIR, 'train_clean.csv'), index=False, encoding='utf-8-sig')
df_dev.to_csv(os.path.join(PROC_DIR, 'dev_clean.csv'), index=False, encoding='utf-8-sig')
df_test.to_csv(os.path.join(PROC_DIR, 'test_clean.csv'), index=False, encoding='utf-8-sig')
df_train.shape, df_dev.shape, df_test.shape

((7776, 16), (1110, 16), (2224, 16))

In [23]:
os.listdir(PROC_DIR)

['actionable_insights.csv',
 'dev_clean.csv',
 'dev_pred_phobert.csv',
 'dev_raw.csv',
 'sample_excel_predicted.xlsx',
 'test_clean.csv',
 'test_pred_baseline.csv',
 'test_pred_phobert.csv',
 'test_raw.csv',
 'train_clean.csv',
 'train_raw.csv',
 'visfd_merged_raw.csv']